# U06 补充: RNN 的结构

> 本节是对 `lesson.ipynb` 中 RNN 部分的补充, 来源: 尚硅谷《大模型技术之 NLP》第三章 3.1 节。
>
> 内容: **基础结构 -> 多层结构 -> 双向结构 -> 多层 + 双向结构**

在 `lesson.ipynb` 里我们已经知道:
- RNN 的核心公式: $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$
- 同一套参数 $W_{xh}, W_{hh}, b$ 在每个时间步重复使用

这一节, 我们换个视角, 看 RNN 的**结构形态**是怎么演化的。

## 1. 基础结构 (Vanilla RNN)

RNN 的核心是一个**带循环连接的隐藏层**, 以**时间步 (time step)** 为单位, 一个 token 一个 token 地处理输入序列。

在每个时间步:
1. 接收当前 token 的向量 $x_t$
2. 接收上一步的隐藏状态 $h_{t-1}$
3. 计算并输出新的隐藏状态 $h_t$, 传给下一步

### 1.1 整体结构

![RNN 基础结构](images/image33.png)

图中可以看到: 同一个 RNN 单元(同一套参数)在时间维度上不断被复用, $h_{t-1}$ 像接力棒一样一步步往后传。

### 1.2 隐藏层内部计算细节

公式展开后, 每个时间步内部做的事情是:

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b)$$

![RNN 计算细节](images/image34.png)

重点:
- $W_{xh}$ 把当前输入投影到隐藏空间
- $W_{hh}$ 把上一时刻的隐藏状态投影到隐藏空间
- 两个投影相加, 再过 tanh -> 当前隐藏状态

### 1.3 简化示意图

因为后面要画更复杂的结构 (多层、双向), 内部细节再画就太挤了, 所以接下来用**简化示意图**: 省略内部细节, 只保留"层 + 时间步"的整体结构。

![RNN 简化示意图](images/image35.png)

请记住这种画法, 后面所有图都基于它。

## 2. 多层结构 (Stacked / Deep RNN)

**问题**: 单层 RNN 只有一个隐藏层, 表达能力有限, 难以同时建模"局部模式"和"高层语义"。

**做法**: 把多个 RNN **纵向堆叠** 起来。

![多层 RNN 结构](images/image36.png)

### 数据是怎么流的

- 最底层 RNN: 输入是**原始序列** $x_1, x_2, \dots, x_T$ (词向量)
- 中间每一层: 输入是**下一层每个时间步的输出序列**
- 最顶层 RNN: 输出作为最终结果, 送给下游任务 (分类头、解码器等)

### 直观理解 (核心假设)

| 层级 | 学到的东西 |
|------|-----------|
| 底层 | 局部模式: 词组、短语搭配 |
| 高层 | 抽象语义: 句子主题、整体语境 |

对应 PyTorch 里就是 `nn.RNN(..., num_layers=N)` 的 `num_layers` 参数。

## 3. 双向结构 (Bidirectional RNN)

### 3.1 单向 RNN 的限制

基础 RNN 在每个时间步只能看到**当前词及它之前的所有词**, 看不到"后文"。

对很多任务这是个明显短板, 比如**序列标注**: 要给每个词打标签时, 只看前文常常不够。

![单向 RNN 在序列标注的局限](images/image37.png)

经典例子: 给 "他/打/了/我/一拳" 中的 "打" 标注词性, 只看 "他" 不一定够; 看到后面的 "一拳" 就更确定它是动词。

### 3.2 双向 RNN 的解法

同一个序列同时跑两个 RNN:

- **正向 RNN**: 按时间顺序 (从前到后) 处理 -> 得到 $\overrightarrow{h_t}$
- **反向 RNN**: 按逆时间顺序 (从后到前) 处理 -> 得到 $\overleftarrow{h_t}$

每个时间步的最终输出 = 正向 + 反向的**组合** (拼接 concat 或者求和):

$$h_t = [\overrightarrow{h_t}\; ;\; \overleftarrow{h_t}]$$

![双向 RNN 结构](images/image38.png)

### 关键点

- 输出维度变成 `2 * hidden_size` (拼接的情况)
- 正反两个方向**参数独立**, 不共享
- 适合**整段序列已知**的场景 (序列标注、命名实体识别、阅读理解)
- **不适合在线生成**: 因为需要看到"后文", 流式生成场景用不了

对应 PyTorch 里就是 `nn.RNN(..., bidirectional=True)`。

## 4. 多层 + 双向结构

把第 2 节和第 3 节叠在一起: **每一层都是双向 RNN**, 然后把多个双向层纵向堆起来。

![多层 + 双向 RNN](images/image39.png)

### 数据流动

- 第 1 层 (最底): 双向 RNN 处理原始词向量序列, 每个时间步输出 $[\overrightarrow{h_t^{(1)}}; \overleftarrow{h_t^{(1)}}]$
- 第 2 层: 把第 1 层每个时间步的拼接输出当作自己的输入, 再来一次双向
- ... 一直到顶层

### PyTorch 代码

```python
rnn = nn.RNN(
    input_size=embed_dim,
    hidden_size=128,
    num_layers=2,        # <- 多层
    bidirectional=True,  # <- 双向
    batch_first=True,
)
```

### 输出形状变化

假设 `batch=4, seq_len=6, hidden_size=128, num_layers=2, bidirectional=True`:

| 张量 | 形状 | 含义 |
|------|------|------|
| `output` | `(4, 6, 128*2) = (4, 6, 256)` | 每个时间步**顶层**的双向拼接输出 |
| `hn`     | `(2*2, 4, 128) = (4, 4, 128)` | `num_layers * num_directions` 个最终隐藏状态 |

注意: `hn` 的第一维是 `num_layers * num_directions`, 顺序是 `[layer0_fw, layer0_bw, layer1_fw, layer1_bw]`。

## 5. 小结

| 结构 | 关键参数 | 解决什么问题 | 输出维度 |
|------|---------|-------------|---------|
| 基础 RNN | `num_layers=1, bidirectional=False` | 建模序列上文 | `hidden_size` |
| 多层 RNN | `num_layers=N` | 提升表达能力, 学习层次化特征 | `hidden_size` |
| 双向 RNN | `bidirectional=True` | 同时利用前文和后文 | `2 * hidden_size` |
| 多层 + 双向 | 两个都开 | 表达力 + 全上下文 | `2 * hidden_size` |

### 选型建议

- **语言模型 / 流式生成**: 只能用单向 (因为没有后文)
- **文本分类 / 情感分析**: 双向更好
- **序列标注 (NER, 词性标注)**: 双向几乎是标配
- **机器翻译 Encoder**: 多层双向 (后面 U10 Seq2Seq 会用到)
- **机器翻译 Decoder**: 单向 (生成是流式的)

下一步去看 `lesson.ipynb` 第 6 节再做练习, 重点理解 `output` 和 `hn` 在双向、多层情况下的形状变化。